# Week 6 · 06 Paper practice: regularization, stability and CNNs

Exam-style practice for the week, in the exam's three forms: computing by hand, writing code by hand, and reading PyTorch code. Part II's code is booklet-level only: two blocks from your booklet (gradient boosting, the decision-tree split) and three of this week's blocks. About 75 minutes on paper; then type Part II and run the checks.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "06")      # defines check(n)

## M. How to use this notebook

Set a timer for 75 minutes and work on paper: Part I (by hand, about 35 minutes), Part II (code by hand, about 25 minutes), Part III (reading PyTorch, about 15 minutes). Then mark yourself, type Part II into the cells and run the checks. Part IV (Week 3–4, spaced) is extra. Log the score and the misses **by topic**.

| Part | What | Points |
|---|---|---|
| I | by hand: sizes, a convolution, batch norm, dropout, weight decay, residuals | 8 × 5 = 40 |
| II | handwritten code (booklet level), then checked | 5 × 8 = 40 |
| III | reading PyTorch code | 5 × 4 = 20 |
| IV | Weeks 3–4, spaced (not scored) |  |

## I. By hand (40 points)

**Drill (on paper).** **I1 · Shapes.** A batch of colour images `(N, 3, 32, 32)` goes through `Conv2d(3, 16, 5)` (no padding), `MaxPool2d(2)`, `Conv2d(16, 32, 3, padding=1, stride=2)`, `MaxPool2d(2)`. Give the shape after each layer, the number of parameters of each layer, and the `in_features` of a final `Linear` to 10 classes.

<details><summary>Worked answer</summary>

Conv1: $\lfloor (32 - 5)/1 \rfloor + 1 = 28$ → `(N, 16, 28, 28)`, $16 \cdot 3 \cdot 25 + 16 = 1216$ parameters. Pool: `(N, 16, 14, 14)`, 0. Conv2: $\lfloor (14 + 2 - 3)/2 \rfloor + 1 = 7$ → `(N, 32, 7, 7)`, $32 \cdot 16 \cdot 9 + 32 = 4640$. Pool: $\lfloor 7/2 \rfloor = 3$ → `(N, 32, 3, 3)`, 0. Flatten: $32 \cdot 3 \cdot 3 = 288$, so `Linear(288, 10)` with $2890$ parameters.

</details>

**Drill (on paper).** **I2 · A convolution and a pool by hand.** $x = \begin{pmatrix} 2 & 0 & 1 & 1 \\ 1 & 3 & 0 & 2 \\ 0 & 1 & 2 & 1 \\ 1 & 0 & 1 & 3 \end{pmatrix}$, kernel $K = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$, stride 1, no padding. (a) The output (size and every entry). (b) Max pooling 2×2 with stride 2 applied to $x$.

<details><summary>Worked answer</summary>

(a) Size $4 - 2 + 1 = 3$. Each output is (top-left of the window) − (bottom-right): $y_{ij} = x_{ij} - x_{i+1, j+1}$. Row 0: $2 - 3 = -1$, $0 - 0 = 0$, $1 - 2 = -1$. Row 1: $1 - 1 = 0$, $3 - 2 = 1$, $0 - 1 = -1$. Row 2: $0 - 0 = 0$, $1 - 1 = 0$, $2 - 3 = -1$. So $\begin{pmatrix} -1 & 0 & -1 \\ 0 & 1 & -1 \\ 0 & 0 & -1 \end{pmatrix}$. (b) The four 2×2 windows' maxima: $\begin{pmatrix} 3 & 2 \\ 1 & 3 \end{pmatrix}$.

</details>

**Drill (on paper).** **I3 · Batch norm.** One feature over a batch of 4: $(2, 4, 6, 8)$, with $\gamma = 2$, $\beta = 1$ (ignore $\varepsilon$). Compute the outputs. Then: which numbers does `model.eval()` use instead of this batch's mean and variance?

<details><summary>Worked answer</summary>

$\mu = 5$, $\sigma^2 = \frac{9 + 1 + 1 + 9}{4} = 5$, $\sigma = 2.236$. Standardized: $(-1.342, -0.447, 0.447, 1.342)$. Times 2 plus 1: $(-1.683, 0.106, 1.894, 3.683)$. In evaluation mode, the running averages of the mean and variance collected during training.

</details>

**Drill (on paper).** **I4 · Dropout.** $p = 0.2$, $h = (5, 10, 0, 2.5)$, and this step's mask keeps units 0, 1 and 3. What does inverted dropout output, and why is that the right scaling?

<details><summary>Worked answer</summary>

$(5, 10, 0, 2.5) \cdot (1, 1, 0, 1) / 0.8 = (6.25, 12.5, 0, 3.125)$. A unit is kept with probability 0.8 and then multiplied by $1/0.8$, so its expected value is $0.8 \cdot h / 0.8 = h$: the next layer sees the same average in training and at test time, where dropout is off.

</details>

**Drill (on paper).** **I5 · Weight decay.** SGD with learning rate $0.1$ and `weight_decay=0.01`. A weight $w = 3$ has gradient $0.5$. Its new value? Rewrite the update as "shrink, then step".

<details><summary>Worked answer</summary>

$w - \eta(g + \lambda w) = 3 - 0.1(0.5 + 0.03) = 2.947$. Shrink: $(1 - 0.1 \cdot 0.01) \cdot 3 = 0.999 \cdot 3 = 2.997$; step: $2.997 - 0.1 \cdot 0.5 = 2.947$.

</details>

**Drill (on paper).** **I6 · Residuals.** Five blocks in a row; in each, the layers' derivative is $F' = 0.2$. By what factor is the gradient multiplied on its way back through all five, without and with skip connections?

<details><summary>Worked answer</summary>

Without: $0.2^5 = 0.00032$. With: $(1 + 0.2)^5 = 2.49$. The skip connection's $+1$ keeps the factor near 1 instead of near 0.

</details>

**Drill (on paper).** **I7 · Initialization.** (a) A ReLU layer has 512 inputs. The standard deviation of He initialization? (b) $z = \sum_{j=1}^{100} w_j x_j$ with independent zero-mean $w_j$ of variance $0.01$ and $x_j$ of variance 4. $\operatorname{Var}(z)$?

<details><summary>Worked answer</summary>

(a) $\sqrt{2/512} = \sqrt{1/256} = 0.0625$. (b) $n\, v_w\, v_x = 100 \cdot 0.01 \cdot 4 = 4$.

</details>

**Drill (on paper).** **I8 · Early stopping.** Validation losses $1.0, 0.8, 0.75, 0.77, 0.74, 0.76, 0.78, 0.79$ (epochs 0–7), patience 2. At which epoch does training stop, and which epoch's weights are kept?

<details><summary>Worked answer</summary>

New bests at epochs 0, 1, 2 and 4 (0.74). Epoch 5: 1 epoch without a new best; epoch 6: 2, which reaches the patience, so it stops at **epoch 6** and keeps **epoch 4**'s weights.

</details>

## II. Handwritten code (40 points)

Exercises 1–2 are booklet blocks (chapters 5 and 3), from memory. Exercises 3–5 are this week's blocks. Write each on paper first; then type it and run its check.

**Exercise 1 · Booklet 5: gradient boosting.** `gboost(X, y, n_trees, lr)`: start from $F_0 = \bar y$; for each tree: residuals $r = y - F$; fit `DecisionTreeRegressor(max_depth=3, random_state=0)` to $(X, r)$; $F \leftarrow F + \eta \cdot$ `tree.predict(X)`. Return `(F0, trees)` with `F0 = y.mean()`. `gb_predict(F0, trees, X, lr)` returns $F_0 + \eta \sum_m h_m(X)$.

**Example**: what goes in, and what should come out:
```python
from sklearn.datasets import load_diabetes
Xd, yd = load_diabetes(return_X_y=True)
F0, trees = gboost(Xd, yd, 300, 0.1)
round(F0, 1), len(trees)     # → (152.1, 300)
```

In [ ]:
from sklearn.tree import DecisionTreeRegressor

def gboost(X, y, n_trees=300, lr=0.1):
    pass  # your code

def gb_predict(F0, trees, X, lr=0.1):
    pass  # your code

In [ ]:
check(1)

**Exercise 2 · Booklet 3: the best split (Gini).** `best_split(X, y)` for integer labels `y`: try every feature $j$ and every midpoint $t$ between its sorted unique values; the gain is $\Delta = G(y) - \big(\frac{n_L}{n} G(y_L) + \frac{n_R}{n} G(y_R)\big)$ with $G(y) = 1 - \sum_k p_k^2$ and $L = \{x_j \le t\}$. Return `(j, t, delta)` for the largest gain, starting from `(None, None, 0.0)` and replacing only on a **strictly** larger gain.

**Example**: what goes in, and what should come out:
```python
Xs = np.array([[1.0], [2.0], [3.0], [4.0]]); ys = np.array([0, 0, 1, 1])
best_split(Xs, ys)     # → (0, 2.5, 0.5)   a perfect split: Gini 0.5 → 0
```

In [ ]:
def gini(y):
    pass  # your code

def best_split(X, y):
    pass  # your code

In [ ]:
check(2)

**Exercise 3 · A convolution with stride and padding.** One channel: pad `x` with `padding` rings of zeros (`np.pad(x, padding)`), then slide the $k \times k$ kernel with the given stride: `out[i, j] = np.sum(x[i*s : i*s+k, j*s : j*s+k] * K)` over the $\lfloor (n + 2p - k)/s \rfloor + 1$ positions in each direction. Checked against `F.conv2d`.

**Example**: what goes in, and what should come out:
```python
conv2d(np.array([[2, 0, 1, 1], [1, 3, 0, 2], [0, 1, 2, 1], [1, 0, 1, 3.0]]), np.array([[1, 0], [0, -1.0]]))
# → [[-1.  0. -1.]
#    [ 0.  1. -1.]
#    [ 0.  0. -1.]]     I2 (a)
```

In [ ]:
def conv2d(x, K, stride=1, padding=0):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · Batch norm in evaluation mode.** Like `03`'s training-mode batch norm, but with the stored statistics: $y = \gamma\, \frac{X - \text{running\_mean}}{\sqrt{\text{running\_var} + \varepsilon}} + \beta$, each `(d,)`. Checked against `nn.BatchNorm1d` in `eval()` mode.

**Example**: what goes in, and what should come out:
```python
batchnorm_eval(np.array([[7.0]]), np.array([5.0]), np.array([4.0]), np.array([1.0]), np.array([0.0]))    # → [[1.]]   (7 - 5) / 2
```

In [ ]:
def batchnorm_eval(X, running_mean, running_var, gamma, beta, eps=1e-5):
    pass  # your code

In [ ]:
check(4)

**Exercise 5 · Training with early stopping (`02`'s block).** Full-batch training: `torch.optim.SGD(model.parameters(), lr=lr)` and `nn.CrossEntropyLoss()`. Each epoch: one step on `(X_tr, y_tr)` in train mode, then the validation loss in eval mode with `no_grad`. Keep the best (lowest) validation loss with a `copy.deepcopy(model.state_dict())`; stop when `epoch - best_epoch >= patience`. At the end, load the best weights into `model` and return `best_epoch`.

**Example**: what goes in, and what should come out:
```python
# on random data the check compares your weights and best epoch with its own run
best_epoch = fit_early_stopping(model, X_tr, y_tr, X_va, y_va, epochs=30, patience=3, lr=0.05)   # → an int
```

In [ ]:
import copy

def fit_early_stopping(model, X_tr, y_tr, X_va, y_va, epochs, patience, lr):
    pass  # your code

In [ ]:
check(5)

In [ ]:
for n in range(1, 6):
    check(n)

## III. Reading PyTorch code (20 points)

**Drill (on paper).** **III1 · Shapes.**\n```python\nx = torch.randn(4, 3, 32, 32)\nconv = nn.Conv2d(3, 6, kernel_size=5)\nprint(conv(x).shape, conv.weight.shape, sum(p.numel() for p in conv.parameters()))\n```

<details><summary>Worked answer</summary>

`torch.Size([4, 6, 28, 28]) torch.Size([6, 3, 5, 5]) 456`: $32 - 5 + 1 = 28$; weight (out, in, k, k); $6 \cdot 3 \cdot 25 + 6 = 456$.

</details>

**Drill (on paper).** **III2 · Find the bug.**\n```python\nclass Net(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)\n        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)\n        self.pool = nn.MaxPool2d(2)\n        self.fc = nn.Linear(16 * 14 * 14, 10)\n    def forward(self, x):                 # x: (N, 1, 28, 28)\n        x = self.pool(torch.relu(self.conv1(x)))\n        x = self.pool(torch.relu(self.conv2(x)))\n        return self.fc(x.flatten(start_dim=1))\n```

<details><summary>Worked answer</summary>

There are **two** pools, so the maps are 7×7, not 14×14: the flattened size is $16 \cdot 7 \cdot 7 = 784$. As written, PyTorch raises a shape error (`mat1 and mat2 shapes cannot be multiplied (N x 784 and 3136 x 10)`). Fix: `nn.Linear(16 * 7 * 7, 10)`. Find this by printing the shape before `fc` (`05` §2).

</details>

**Drill (on paper).** **III3 · Why does this fail?**\n```python\nmodel = nn.Sequential(nn.Linear(10, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Linear(32, 2))\nout = model(torch.randn(1, 10))     # one sample, model still in training mode\n```

<details><summary>Worked answer</summary>

In training mode batch norm normalizes with the **batch's** mean and variance, and a batch of one sample has variance 0 for every feature, so PyTorch refuses (`Expected more than 1 value per channel when training`). Call `model.eval()` to predict single samples (it then uses the running averages), or use `nn.LayerNorm`, which normalizes each sample on its own.

</details>

**Drill (on paper).** **III4 · The order.** Where does `nn.utils.clip_grad_norm_(model.parameters(), 1.0)` go in the five-line loop, and why exactly there?

<details><summary>Worked answer</summary>

After `loss.backward()` and before `opt.step()`: `backward()` is what fills the gradients, and `step()` is what uses them. Clipping before `backward()` would clip nothing; after `step()`, too late.

</details>

**Drill (on paper).** **III5 · Adam vs AdamW.** `torch.optim.Adam(params, lr=1e-3, weight_decay=0.01)` vs `torch.optim.AdamW(params, lr=1e-3, weight_decay=0.01)`: what's the difference?

<details><summary>Worked answer</summary>

`Adam` adds $\lambda w$ to the gradient, which Adam then divides by $\sqrt{\hat s}$, so weights with large gradients are barely decayed. `AdamW` shrinks the weights directly ($w \leftarrow w - \eta\lambda w$), separately from the Adam step, so every weight decays at the same rate. AdamW is the one to use (`02` §2).

</details>

## IV. Weeks 3–4, spaced (not scored)

**Drill (on paper).** **IV1 · AdaBoost.** A stump's weighted error is $\varepsilon = 0.2$. Its vote $\alpha$? By what factor is the weight of a misclassified point multiplied (before normalizing)?

<details><summary>Worked answer</summary>

$\alpha = \frac{1}{2}\ln\frac{0.8}{0.2} = \frac{1}{2}\ln 4 = 0.693$. Misclassified points: $\times e^{\alpha} = \times 2$; correct ones: $\times e^{-\alpha} = \times 0.5$.

</details>

**Drill (on paper).** **IV2 · Gini.** A node with 3 points of class A and 1 of class B. Its Gini impurity?

<details><summary>Worked answer</summary>

$1 - (0.75^2 + 0.25^2) = 1 - 0.625 = 0.375$.

</details>

**Drill (on paper).** **IV3 · PCA.** The covariance matrix has eigenvalues 4 and 1. What fraction of the variance does PC1 keep? What is $u^\top S u$ for the unit vector $u$ along PC1?

<details><summary>Worked answer</summary>

$\frac{4}{4 + 1} = 0.8$. $u^\top S u = \lambda_1 = 4$ (your Week 4 proposition: the variance of the projections onto $u$).

</details>

---
Mark Parts I–III, then log the score and every miss by topic. Solutions for Part II: `solutions/06_solutions.py`. Next: `07_sunday_quiz.ipynb`.